# Feature Engineering & Preprocessing

## Project Overview

This notebook focuses on preparing the Anti-Money Laundering (AML) dataset for machine learning model development.

The main objectives are:
- Perform feature engineering.
- Handle numerical and categorical features.
- Investigate potential target leakage.
- Define the input features and target variable.
- Split the dataset into training, validation, and testing sets.
- Build a preprocessing pipeline.
- Save the prepared datasets and preprocessing configuration for model training.

**Note:** Model training and evaluation are performed in a separate notebook.

## 1. Import Libraries

Import the required Python libraries for data manipulation, numerical operations, preprocessing, dataset splitting, and saving the prepared artifacts.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import json
import joblib
import zipfile

from pathlib import Path
from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

## 2. Load Cleaned Dataset

Load the cleaned AML dataset generated during the Data Understanding and Exploratory Data Analysis stage.

This dataset serves as the starting point for feature engineering and preprocessing.

The feature-engineered dataset will be generated within this notebook.

In [ ]:
uploaded = files.upload()

file_name = next(iter(uploaded))

df = pd.read_csv(file_name)

print("Cleaned dataset loaded successfully!")
print("Dataset shape:", df.shape)

df.head()

Saving money_laundring_clean.csv to money_laundring_clean (2).csv
Cleaned dataset loaded successfully!
Dataset shape: (97066, 12)


,Time,Date,Sender_account,Receiver_account,Amount,Payment_currency,Received_currency,Sender_bank_location,Receiver_bank_location,Payment_type,Is_laundering,Laundering_type
0,10:51:48,2022-10-07,3884364796,3678154204,92.61,Euros,UK pounds,UK,UK,Credit card,0,Normal_Small_Fan_Out
1,19:44:48,2022-10-11,7900125583,3009831077,5157.16,UK pounds,UK pounds,UK,Japan,Cross-border,1,Behavioural_Change_2
2,8:17:22,2022-10-15,8175367805,3447405915,7426.99,UK pounds,UK pounds,UAE,UK,Cheque,0,Normal_Fan_Out
3,15:04:00,2022-10-18,786553572,6980069974,3391.18,UK pounds,UK pounds,US,Nigeria,ACH,0,Normal_Small_Fan_Out
4,4:48:34,2022-10-29,2839583258,1512033381,6951.29,US dollars,US dollars,UK,Germany,Debit card,0,Normal_Fan_In


## 3. Validate Dataset Structure

Verify that the cleaned dataset contains all expected columns before performing feature engineering.

This step also checks duplicate records and missing values to identify potential data quality issues.

In [ ]:
# Display actual dataset information

print("Dataset shape:", df.shape)

print("\nActual column names:")
for col in df.columns:
    print(repr(col))

print("\nFirst 5 rows:")
display(df.head())

print("\nData types:")
print(df.dtypes)

Dataset shape: (97066, 12)

Actual column names:
'Time'
'Date'
'Sender_account'
'Receiver_account'
'Amount'
'Payment_currency'
'Received_currency'
'Sender_bank_location'
'Receiver_bank_location'
'Payment_type'
'Is_laundering'
'Laundering_type'

First 5 rows:


,Time,Date,Sender_account,Receiver_account,Amount,Payment_currency,Received_currency,Sender_bank_location,Receiver_bank_location,Payment_type,Is_laundering,Laundering_type
0,10:51:48,2022-10-07,3884364796,3678154204,92.61,Euros,UK pounds,UK,UK,Credit card,0,Normal_Small_Fan_Out
1,19:44:48,2022-10-11,7900125583,3009831077,5157.16,UK pounds,UK pounds,UK,Japan,Cross-border,1,Behavioural_Change_2
2,8:17:22,2022-10-15,8175367805,3447405915,7426.99,UK pounds,UK pounds,UAE,UK,Cheque,0,Normal_Fan_Out
3,15:04:00,2022-10-18,786553572,6980069974,3391.18,UK pounds,UK pounds,US,Nigeria,ACH,0,Normal_Small_Fan_Out
4,4:48:34,2022-10-29,2839583258,1512033381,6951.29,US dollars,US dollars,UK,Germany,Debit card,0,Normal_Fan_In



Data types:
Time                       object
Date                       object
Sender_account              int64
Receiver_account            int64
Amount                    float64
Payment_currency           object
Received_currency          object
Sender_bank_location       object
Receiver_bank_location     object
Payment_type               object
Is_laundering               int64
Laundering_type            object
dtype: object


In [ ]:
missing_columns = set(expected_columns) - set(df.columns)

if missing_columns:
    print("Missing columns:", missing_columns)
else:
    print("All expected columns are available!")

print("\nDataset shape:", df.shape)

Missing columns: {'Reciever_frequancy', 'is_bank_same_location', 'is_payment_same', 'Day', 'hour', 'Sender_frequancy'}

Dataset shape: (97066, 12)


## 4. Feature Engineering

Feature Engineering transforms existing variables into more informative representations that can help machine learning models identify patterns in transaction behavior.

In this section, we create cyclical time features and validate numerical and Boolean variables.

All original columns are preserved, and the new engineered features are added to the dataset.

### 4.1 Cyclical Encoding of Hour

The `hour` feature represents the transaction time from 0 to 23.

Since time is cyclical, hour 23 is close to hour 0. Treating it as a regular numerical feature may incorrectly represent the relationship between these values.

To preserve this cyclical relationship, we apply Sine and Cosine transformations, generating two new features:

- `hour_sin`
- `hour_cos`

The original `hour` column is retained in the dataset.

In [ ]:
# Extract hour from Time

df["Time"] = pd.to_datetime(
    df["Time"].astype(str).str.strip(),
    format="%H:%M:%S",
    errors="coerce"
)

if df["Time"].isna().any():
    raise ValueError("Invalid or missing Time values found.")

df["hour"] = df["Time"].dt.hour

print("Hour extracted successfully!")
display(df[["Time", "hour"]].head())

Hour extracted successfully!


,Time,hour
0,1900-01-01 10:51:48,10
1,1900-01-01 19:44:48,19
2,1900-01-01 08:17:22,8
3,1900-01-01 15:04:00,15
4,1900-01-01 04:48:34,4


In [ ]:
df["hour"] = pd.to_numeric(df["hour"], errors="coerce")

if df["hour"].isna().any():
    raise ValueError("Invalid or missing hour values found.")

if not df["hour"].between(0, 23).all():
    raise ValueError("Hour values must be between 0 and 23.")

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

print("Cyclical hour features created successfully!")

df[["hour", "hour_sin", "hour_cos"]].head()

Cyclical hour features created successfully!


,hour,hour_sin,hour_cos
0,10,0.500000,-0.866025
1,19,-0.965926,0.258819
2,8,0.866025,-0.500000
3,15,-0.707107,-0.707107
4,4,0.866025,0.500000


### 4.2 Amount Feature Validation

The `Amount` feature represents the transaction value.

We convert it to a numerical data type to ensure compatibility with machine learning algorithms.

Negative values are checked separately instead of being automatically removed or modified.

In [ ]:
df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")

print("Missing Amount values:", df["Amount"].isna().sum())
print("Negative Amount values:", (df["Amount"] < 0).sum())

if (df["Amount"] < 0).any():
    raise ValueError("Negative transaction amounts require investigation.")

Missing Amount values: 0
Negative Amount values: 0


### 4.3 Boolean Feature Validation

The dataset contains Boolean features describing relationships between transaction properties.

These features are represented numerically using binary values:

- `0`: False
- `1`: True

This ensures consistent numerical representation during preprocessing and model training.

In [ ]:
# Create boolean features

df["is_payment_same"] = (
    df["Payment_currency"] == df["Received_currency"]
).astype(int)

df["is_bank_same_location"] = (
    df["Sender_bank_location"] == df["Receiver_bank_location"]
).astype(int)

print("Boolean features created successfully!")

display(df[[
    "Payment_currency",
    "Received_currency",
    "is_payment_same",
    "Sender_bank_location",
    "Receiver_bank_location",
    "is_bank_same_location"
]].head())

Boolean features created successfully!


,Payment_currency,Received_currency,is_payment_same,Sender_bank_location,Receiver_bank_location,is_bank_same_location
0,Euros,UK pounds,0,UK,UK,1
1,UK pounds,UK pounds,1,UK,Japan,0
2,UK pounds,UK pounds,1,UAE,UK,0
3,UK pounds,UK pounds,1,US,Nigeria,0
4,US dollars,US dollars,1,UK,Germany,0


In [ ]:
boolean_features = [
    "is_payment_same",
    "is_bank_same_location"
]

for col in boolean_features:
    df[col] = pd.to_numeric(df[col], errors="coerce")

    if not df[col].dropna().isin([0, 1]).all():
        raise ValueError(f"Unexpected values found in {col}.")

print("Boolean features validated successfully!")

print(df[boolean_features].dtypes)

Boolean features validated successfully!
is_payment_same          int64
is_bank_same_location    int64
dtype: object


### 4.4 Verify Engineered Features

Review the generated cyclical features and verify that the dataset contains all original columns along with the newly created features.

In [ ]:
print("Dataset shape after feature engineering:", df.shape)

print("\nDataset columns:")
print(df.columns.tolist())

df.head()


Dataset shape after feature engineering: (97066, 17)

Dataset columns:
['Time', 'Date', 'Sender_account', 'Receiver_account', 'Amount', 'Payment_currency', 'Received_currency', 'Sender_bank_location', 'Receiver_bank_location', 'Payment_type', 'Is_laundering', 'Laundering_type', 'hour', 'hour_sin', 'hour_cos', 'is_payment_same', 'is_bank_same_location']


,Time,Date,Sender_account,Receiver_account,Amount,Payment_currency,Received_currency,Sender_bank_location,Receiver_bank_location,Payment_type,Is_laundering,Laundering_type,hour,hour_sin,hour_cos,is_payment_same,is_bank_same_location
0,1900-01-01 10:51:48,2022-10-07,3884364796,3678154204,92.61,Euros,UK pounds,UK,UK,Credit card,0,Normal_Small_Fan_Out,10,0.500000,-0.866025,0,1
1,1900-01-01 19:44:48,2022-10-11,7900125583,3009831077,5157.16,UK pounds,UK pounds,UK,Japan,Cross-border,1,Behavioural_Change_2,19,-0.965926,0.258819,1,0
2,1900-01-01 08:17:22,2022-10-15,8175367805,3447405915,7426.99,UK pounds,UK pounds,UAE,UK,Cheque,0,Normal_Fan_Out,8,0.866025,-0.500000,1,0
3,1900-01-01 15:04:00,2022-10-18,786553572,6980069974,3391.18,UK pounds,UK pounds,US,Nigeria,ACH,0,Normal_Small_Fan_Out,15,-0.707107,-0.707107,1,0
4,1900-01-01 04:48:34,2022-10-29,2839583258,1512033381,6951.29,US dollars,US dollars,UK,Germany,Debit card,0,Normal_Fan_In,4,0.866025,0.500000,1,0


## 5. Target Leakage Investigation

Target Leakage occurs when information that would not legitimately be available at prediction time is included in the model features.

The `Laundering_type` column describes the type of laundering activity and may directly reveal whether a transaction is suspicious.

We investigate its relationship with the target variable, `Is_laundering`, to determine whether it should be excluded from model inputs.

In [ ]:
leakage_analysis = pd.crosstab(
    df["Laundering_type"],
    df["Is_laundering"],
    normalize="index"
)

leakage_analysis

Is_laundering,0,1
Laundering_type,,
Behavioural_Change_1,0.000000,1.000000
Behavioural_Change_2,0.000000,1.000000
Bipartite,0.000000,1.000000
Cash_Withdrawal,0.000000,1.000000
Cycle,0.000000,1.000000
Deposit-Send,0.000000,1.000000
Fan_In,0.000000,1.000000
Fan_Out,0.000000,1.000000
Gather-Scatter,0.000000,1.000000


### Leakage Prevention

Based on the relationship between `Laundering_type` and the target variable, this column is excluded from model inputs to prevent target leakage.

The original column is preserved in the feature-engineered dataset for reference and analysis.

## 6. Feature Selection

After feature engineering and target leakage investigation, select the input features that will be used by machine learning models.

The following columns are excluded from model inputs:

- `Is_laundering`: Target variable, not an input feature.
- `Laundering_type`: Excluded to prevent target leakage.
- `hour`: Replaced by `hour_sin` and `hour_cos`.
- `Sender_frequancy`: Excluded based on the project's current feature-selection decision.
- `Reciever_frequancy`: Excluded based on the project's current feature-selection decision.

The remaining features are used to construct X, while `Is_laundering` is used as y.

In [ ]:
# Create transaction frequency features

df["Sender_frequancy"] = (
    df.groupby("Sender_account")["Sender_account"].transform("count")
)

df["Reciever_frequancy"] = (
    df.groupby("Receiver_account")["Receiver_account"].transform("count")
)

print("Frequency features created successfully!")

display(df[[
    "Sender_account",
    "Sender_frequancy",
    "Receiver_account",
    "Reciever_frequancy"
]].head())

Frequency features created successfully!


,Sender_account,Sender_frequancy,Receiver_account,Reciever_frequancy
0,3884364796,5,3678154204,1
1,7900125583,110,3009831077,25
2,8175367805,14,3447405915,3
3,786553572,2,6980069974,1
4,2839583258,2,1512033381,20


In [ ]:
target_column = "Is_laundering"

excluded_features = [
    "Is_laundering",
    "Laundering_type",
    "hour",
    "Sender_frequancy",
    "Reciever_frequancy"
]

X = df.drop(columns=excluded_features)
y = df[target_column]

if y.isna().any():
    raise ValueError("Missing values found in the target variable.")

if not y.isin([0, 1]).all():
    raise ValueError("Target variable must contain only 0 and 1.")

print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("\nSelected features:")
print(X.columns.tolist())

print("\nTarget distribution:")
print(y.value_counts())

Features shape: (97066, 14)
Target shape: (97066,)

Selected features:
['Time', 'Date', 'Sender_account', 'Receiver_account', 'Amount', 'Payment_currency', 'Received_currency', 'Sender_bank_location', 'Receiver_bank_location', 'Payment_type', 'hour_sin', 'hour_cos', 'is_payment_same', 'is_bank_same_location']

Target distribution:
Is_laundering
0    74585
1    22481
Name: count, dtype: int64


### 6.2 Identify Numerical and Categorical Features

The selected features are divided into two groups to apply appropriate preprocessing techniques.

**Categorical Features:**
- Day
- Payment_currency
- Received_currency
- Sender_bank_location
- Receiver_bank_location
- Payment_type

**Numerical Features:**
- Amount
- is_payment_same
- is_bank_same_location
- hour_sin
- hour_cos

In [ ]:
# Extract day of the week from Date

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

if df["Date"].isna().any():
    raise ValueError("Invalid or missing Date values found.")

df["Day"] = df["Date"].dt.day_name()

print("Day feature created successfully!")

display(df[["Date", "Day"]].head())

print("\nDay distribution:")
print(df["Day"].value_counts())

Day feature created successfully!


,Date,Day
0,2022-10-07,Friday
1,2022-10-11,Tuesday
2,2022-10-15,Saturday
3,2022-10-18,Tuesday
4,2022-10-29,Saturday



Day distribution:
Day
Friday       15759
Thursday     14269
Saturday     14117
Monday       13674
Wednesday    13298
Tuesday      13297
Sunday       12652
Name: count, dtype: int64


In [ ]:
categorical_features = [
    "Day",
    "Payment_currency",
    "Received_currency",
    "Sender_bank_location",
    "Receiver_bank_location",
    "Payment_type"
]

numerical_features = [
    "Amount",
    "is_payment_same",
    "is_bank_same_location",
    "hour_sin",
    "hour_cos"
]

assert set(categorical_features + numerical_features) == set(X.columns)

print("Categorical features:", len(categorical_features))
print("Numerical features:", len(numerical_features))
print("Total selected features:", X.shape[1])

Categorical features: 6
Numerical features: 5
Total selected features: 11


## 7. Train, Validation, and Test Split

Split the dataset into three independent subsets:

- Training Set: 60% of the data, used to train machine learning models.
- Validation Set: 20% of the data, used for model selection and threshold optimization.
- Test Set: 20% of the data, used for final model evaluation.

Stratified splitting is applied to preserve the target class distribution across all subsets.

A fixed random state of 42 ensures reproducibility.

In [ ]:
X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_dev,
    y_dev,
    test_size=0.25,
    random_state=42,
    stratify=y_dev
)

print("Training set:", X_train.shape, y_train.shape)
print("Validation set:", X_val.shape, y_val.shape)
print("Testing set:", X_test.shape, y_test.shape)

Training set: (58239, 11) (58239,)
Validation set: (19413, 11) (19413,)
Testing set: (19414, 11) (19414,)


### 7.1 Check Target Distribution

Verify that the proportions of legitimate and laundering transactions remain consistent across the training, validation, and testing sets.

This helps ensure that each subset represents the original target distribution.

In [ ]:
distribution = pd.DataFrame({
    "Train": y_train.value_counts(normalize=True),
    "Validation": y_val.value_counts(normalize=True),
    "Test": y_test.value_counts(normalize=True)
}).rename(index={
    0: "Legitimate",
    1: "Laundering"
})

distribution

,Train,Validation,Test
Is_laundering,,,
Legitimate,0.768385,0.768403,0.768415
Laundering,0.231615,0.231597,0.231585


## 8. Build Preprocessing Pipeline

Machine learning algorithms require numerical input and may not handle missing values directly.

A ColumnTransformer is used to apply different preprocessing operations to numerical and categorical features.

**Numerical Features:**
- Missing values are handled using median imputation.

**Categorical Features:**
- Missing values are handled using most frequent value imputation.
- One-Hot Encoding converts categorical values into numerical representations.
- `handle_unknown="ignore"` allows the pipeline to handle previously unseen categories.

The preprocessing pipeline is intentionally left unfitted.

It will be fitted on the training data inside each model pipeline to prevent data leakage from validation and test sets.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

print("Preprocessing pipeline created successfully!")

Preprocessing pipeline created successfully!


## 9. Save Prepared Datasets and Artifacts

Save the training, validation, and testing datasets separately, together with the preprocessing pipeline and feature configuration.

These artifacts will be used directly in the model training notebook, avoiding the need to repeat feature engineering, feature selection, and data splitting.

In [ ]:
output_dir = Path("aml_preprocessing_artifacts")
output_dir.mkdir(exist_ok=True)

# Save feature sets
X_train.to_csv(output_dir / "X_train.csv", index=False)
X_val.to_csv(output_dir / "X_val.csv", index=False)
X_test.to_csv(output_dir / "X_test.csv", index=False)

# Save target variables
y_train.to_frame(name=target_column).to_csv(
    output_dir / "y_train.csv", index=False
)

y_val.to_frame(name=target_column).to_csv(
    output_dir / "y_val.csv", index=False
)

y_test.to_frame(name=target_column).to_csv(
    output_dir / "y_test.csv", index=False
)

# Save the unfitted preprocessing pipeline
joblib.dump(preprocessor, output_dir / "preprocessor.joblib")

# Save feature configuration
feature_config = {
    "target_column": target_column,
    "categorical_features": categorical_features,
    "numerical_features": numerical_features,
    "excluded_features": excluded_features,
    "random_state": 42,
    "split_ratio": {
        "train": 0.60,
        "validation": 0.20,
        "test": 0.20
    }
}

with open(output_dir / "feature_config.json", "w") as f:
    json.dump(feature_config, f, indent=4)

print("All preprocessing artifacts saved successfully!")

All preprocessing artifacts saved successfully!


## 10. Export Feature-Engineered Dataset

Export the complete feature-engineered dataset, preserving all original columns and adding the two cyclical time features.

This file provides a complete record of the dataset after feature engineering, before model-specific feature selection.

In [ ]:
feature_engineered_file = "money_laundring_feature_engineered.csv"

df.to_csv(feature_engineered_file, index=False)

print("Feature-engineered dataset saved successfully!")
print("Dataset shape:", df.shape)

files.download(feature_engineered_file)

Feature-engineered dataset saved successfully!
Dataset shape: (97066, 20)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 11. Export Preprocessing Artifacts

Compress the prepared training, validation, and testing datasets together with the preprocessing pipeline and feature configuration into one ZIP archive.

This archive will be uploaded into the model training notebook.

In [ ]:
zip_filename = "aml_preprocessing_artifacts.zip"

with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file_path in output_dir.iterdir():
        zipf.write(file_path, arcname=file_path.name)

print("Preprocessing archive created successfully!")

files.download(zip_filename)

Preprocessing archive created successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 12. Handoff to Model Training and Evaluation

The feature engineering and preprocessing stages are now complete.

The generated ZIP archive contains:

- X_train.csv
- X_val.csv
- X_test.csv
- y_train.csv
- y_val.csv
- y_test.csv
- preprocessor.joblib
- feature_config.json

These files will be loaded into `Model_Training_and_Evaluation_Organized.ipynb`.

The preprocessing pipeline will be fitted separately within each model pipeline using only the training data.